# Notebook 03 — Joint Training
**Phase 2-3:** Freeze decoder, add LSTM head, train with joint loss.
L = MSE(reconstruction) + lambda * CrossEntropy(classification)

## 0. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.config import (
    LAMBDA, JOINT_EPOCHS, MODELS_DIR, PLOTS_DIR, SEED, CLASSES
)
from src.model import Encoder, Decoder, LSTMHead, JointModel, freeze_decoder
from src.dataset import load_npy_split
from src.train import joint_train, plot_joint_loss, joint_train_lambda_sweep

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch: {torch.__version__}  Device: {DEVICE}")

PyTorch: 2.13.0+cu130  Device: cuda


## 1. Load Data

In [2]:
X_train, y_train = load_npy_split("train")
X_val,   y_val   = load_npy_split("val")

print(f"Train: {X_train.shape}  labels: {np.bincount(y_train)}")
print(f"Val  : {X_val.shape}   labels: {np.bincount(y_val)}")

Train: (10500, 8, 64, 64, 3)  labels: [2100 2100 2100 2100 2100]
Val  : (2250, 8, 64, 64, 3)   labels: [450 450 450 450 450]


## 2. Load Pre-trained Weights and Freeze Decoder

In [3]:
encoder = Encoder()
decoder = Decoder()
encoder.load_state_dict(torch.load(MODELS_DIR / "encoder_pretrained.pth", map_location=DEVICE))
decoder.load_state_dict(torch.load(MODELS_DIR / "decoder_pretrained.pth", map_location=DEVICE))

freeze_decoder(decoder)

trainable = sum(p.numel() for p in decoder.parameters() if p.requires_grad)
total     = sum(p.numel() for p in decoder.parameters())
print(f"Decoder trainable: {trainable}  frozen: {total - trainable}")

[INFO] Decoder frozen (8 parameter tensors).
Decoder trainable: 0  frozen: 2270819


## 3. Build Joint Model

In [4]:
lstm_head   = LSTMHead()
joint_model = JointModel(encoder, decoder, lstm_head)

trainable_total = sum(p.numel() for p in joint_model.parameters() if p.requires_grad)
print(f"Joint model trainable parameters: {trainable_total:,}")

Joint model trainable parameters: 2,396,869


## 4. Joint Training (Lambda = 0.5)

In [ ]:
history = joint_train(joint_model, X_train, y_train, X_val, y_val,
                      lam=LAMBDA, device=DEVICE)

Epoch 01/30  loss=0.3040 recon=0.0362 cls=0.5355 acc=0.816 | val_loss=0.1177 val_acc=0.951
Epoch 02/30  loss=0.0891 recon=0.0366 cls=0.1050 acc=0.968 | val_loss=0.0826 val_acc=0.971
Epoch 03/30  loss=0.0727 recon=0.0373 cls=0.0708 acc=0.978 | val_loss=0.0692 val_acc=0.980
Epoch 04/30  loss=0.0571 recon=0.0368 cls=0.0407 acc=0.988 | val_loss=0.0808 val_acc=0.966
Epoch 05/30  loss=0.0502 recon=0.0366 cls=0.0273 acc=0.992 | val_loss=0.0509 val_acc=0.990
Epoch 06/30  loss=0.0451 recon=0.0357 cls=0.0186 acc=0.994 | val_loss=0.0531 val_acc=0.989
Epoch 07/30  loss=0.0428 recon=0.0346 cls=0.0163 acc=0.994 | val_loss=0.0514 val_acc=0.991
Epoch 08/30  loss=0.0460 recon=0.0347 cls=0.0226 acc=0.993 | val_loss=0.0558 val_acc=0.991
Epoch 09/30  loss=0.0430 recon=0.0347 cls=0.0166 acc=0.995 | val_loss=0.0547 val_acc=0.990
Epoch 10/30  loss=0.0382 recon=0.0336 cls=0.0092 acc=0.997 | val_loss=0.0435 val_acc=0.994
Epoch 11/30  loss=0.0372 recon=0.0324 cls=0.0095 acc=0.997 | val_loss=0.0463 val_acc=0.992

## 5. Loss and Accuracy Curves

In [ ]:
plot_joint_loss(history, save_path=PLOTS_DIR / "joint_loss.png")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(history["train_acc"], label="Train Accuracy", linewidth=1.5)
ax.plot(history["val_acc"],   label="Val Accuracy",   linewidth=1.5, linestyle="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.set_title("Classification Accuracy During Joint Training", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "joint_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Lambda Sensitivity Analysis

In [ ]:
print("\nRunning lambda sensitivity analysis (trains 3 models)...")
lambda_results = joint_train_lambda_sweep(
    X_train, y_train, X_val, y_val,
    lambdas=(0.1, 0.5, 1.0), device=DEVICE
)

lambdas   = list(lambda_results.keys())
val_losses = [lambda_results[l]["val_loss"] for l in lambdas]
val_accs   = [lambda_results[l]["val_acc"]  for l in lambdas]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
ax1.bar([str(l) for l in lambdas], val_losses, color="#4472C4")
ax1.set_title("Val Loss by Lambda", fontweight="bold")
ax1.set_xlabel("Lambda"); ax1.set_ylabel("Validation Loss")
ax2.bar([str(l) for l in lambdas], val_accs, color="#ED7D31")
ax2.set_title("Val Accuracy by Lambda", fontweight="bold")
ax2.set_xlabel("Lambda"); ax2.set_ylabel("Validation Accuracy")
plt.suptitle("Lambda Sensitivity Analysis", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "lambda_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Summary

In [ ]:
joint_model.load_state_dict(
    torch.load(MODELS_DIR / "joint_model_best.pth", map_location=DEVICE))
print(f"Best val accuracy: {max(history['val_acc']):.4f}")
print(f"Best val loss    : {min(history['val_loss']):.4f}")
print("\n[DONE] Joint model saved: joint_model_best.pth")